# The streets of a city

This notebook loads the streets of Boston where a car may drive, with their turn restrictions, and computes routes on them. The data comes from the Overpass API of OpenStreetMap, a file of about 18 MB under the [Open Database License (ODbL 1.0)](https://opendatacommons.org/licenses/odbl/1-0/). The run takes about one minute and about 2 GB of memory.

An OpenStreetMap **way** is a line of a road, of a railway or of another feature. Its tag `highway` gives the kind of road: `motorway`, `primary`, `residential` and others. A **junction** is a point where two or more streets meet. A **turn restriction** forbids a turn at a junction, for example a left turn.

1. Download the streets of Boston.
2. Load them twice: with the turn restrictions, and with any turn allowed.
3. Compute a route on each network and draw them.
4. Count how often the turn restrictions change a route.
5. Find the fastest route with a metric of minutes.

GitHub does not show the maps of a notebook. To see them, run the notebook, or open it in [nbviewer](https://nbviewer.org/github/connor-makowski/scgraph/blob/main/examples/streets_of_a_city.ipynb).

## Setup

`load_from_osm` reads OpenStreetMap files with `osmium`, which the extra `osm` of scgraph installs. The map needs [folium](https://python-visualization.github.io/folium/).

In [1]:
# Until scgraph 4.0 is on PyPI, install it from the `dev` branch:
# %pip install "scgraph[osm] @ git+https://github.com/connor-makowski/scgraph@dev" folium
import random
import tempfile
from collections import Counter
from pathlib import Path

import folium
import requests

from scgraph import GeoGraph
from scgraph.helpers.osm import get_way_direction

## 1. Download the streets

The [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API) answers queries on the data of OpenStreetMap. The query below asks for three things:

- the ways inside the boundary of the city of Boston, with a `highway` value of a street that a car may take,
- the turn restrictions of these ways: OpenStreetMap stores each one as a **relation**, a record with three members: the way that the driver comes from (`from`), the junction (`via`), and the way that the turn leads to (`to`),
- the points of these ways.

The list of `highway` values leaves out the `service` ways, such as the lanes of a car park or a driveway.

The next cell writes the answer, about 18 MB, into the temporary folder of your system. A later run finds the file there and downloads nothing. The data is © OpenStreetMap contributors, under the ODbL 1.0: give this credit with your results.

To get every way with a `highway` tag, footways and paths included, replace the line `way[...]` of the query by `way["highway"](area.city)->.streets;`. The file is then about 47 MB.

In [2]:
# The `highway` values of the streets that a car may take
STREETS = {
    "motorway", "trunk", "primary", "secondary", "tertiary", "unclassified", "residential", "living_street",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
}  # fmt: skip

query = f"""[out:xml][timeout:180];
area["boundary"="administrative"]["admin_level"="8"]["name"="Boston"]["wikidata"="Q100"]->.city;
way["highway"~"^({"|".join(sorted(STREETS))})$"](area.city)->.streets;
rel["type"="restriction"](bw.streets)->.restrictions;
(.streets; .restrictions;);
(._; node(w););
out;"""

streets_file = Path(tempfile.gettempdir()) / "scgraph_examples" / "boston_streets.osm"
if not streets_file.exists():
    # The server refuses a request that does not name its client
    answer = requests.post(
        "https://overpass-api.de/api/interpreter",
        data={"data": query},
        headers={"User-Agent": "scgraph-example-notebook"},
        timeout=300,
    )
    answer.raise_for_status()
    streets_file.parent.mkdir(exist_ok=True)
    partial_file = streets_file.with_suffix(".part")
    partial_file.write_bytes(answer.content)
    # A busy server can answer with an error remark and no data: the file then keeps its temporary name
    if b"<way " not in partial_file.read_bytes():
        raise RuntimeError(f"The answer of the server holds no way, see {partial_file}. Run this cell again later.")
    partial_file.rename(streets_file)
print(f"{streets_file.name}: {streets_file.stat().st_size / 1e6:.1f} MB")

boston_streets.osm: 18.2 MB


The public Overpass server refuses a request when it is busy. Then run the cell again later, or replace its URL by the URL of another server, such as `https://overpass.kumi.systems/api/interpreter`.

## 2. Load the streets

`select` chooses the ways of the file. It receives the tags of a way, and returns its direction, or None to leave the way out. The direction is 1 or -1 for a one-way street, and 0 for a street in both directions. `get_way_direction` reads it from the tags.

The cell loads the file twice. `load_from_osm` reads the turn restrictions by default. `load_turn_restrictions=False` gives the same streets, where a route may make any turn.

In [3]:
def select_streets(tags):
    """The direction of a street, and None for any other way."""
    return get_way_direction(tags) if tags.get("highway") in STREETS else None


streets = GeoGraph.load_from_osm(streets_file, select=select_streets)
streets_any_turn = GeoGraph.load_from_osm(streets_file, select=select_streets, load_turn_restrictions=False)
print(f"Any turn:                   {len(streets_any_turn.nodes):,} nodes")
print(f"With the turn restrictions: {len(streets.nodes):,} nodes")

Any turn:                   14,258 nodes
With the turn restrictions: 14,825 nodes


The network with the turn restrictions has more nodes. A junction with a turn restriction has more than one node, at the same point and with the same OpenStreetMap id. `external_node_ids` gives the OpenStreetMap id of each node, so the next cell counts these junctions.

In [4]:
nodes_by_osm_id = Counter(streets.external_node_ids)
junctions = sum(1 for count in nodes_by_osm_id.values() if count > 1)
print(f"{junctions:,} junctions have more than one node")

444 junctions have more than one node


## 3. A route from Copley Square to Logan Airport

The cell computes the same route on the two networks. Compare the two lengths, and the numbers of points.

In [5]:
COPLEY_SQUARE = {"latitude": 42.3499, "longitude": -71.0774}
LOGAN_AIRPORT = {"latitude": 42.3656, "longitude": -71.0096}

routes = {}
for title, geograph in [("Any turn", streets_any_turn), ("With the turn restrictions", streets)]:
    routes[title] = geograph.get_shortest_path(origin_node=COPLEY_SQUARE, destination_node=LOGAN_AIRPORT)
    print(f"{title}: {routes[title]['length']:.3f} km, {len(routes[title]['coordinate_path']):,} points")

Any turn: 7.844 km, 449 points
With the turn restrictions: 7.914 km, 459 points


The map draws the two routes, each in its own layer. The control at the top right shows or hides a layer.

In [6]:
ROUTE_COLORS = {"Any turn": "#0b0b0b", "With the turn restrictions": "#eb6834"}

route_map = folium.Map(location=[42.357, -71.042], zoom_start=13)
for title, route in routes.items():
    layer = folium.FeatureGroup(name=f"{title}: {route['length']:.3f} km")
    folium.PolyLine(route["coordinate_path"], color=ROUTE_COLORS[title], weight=4).add_to(layer)
    layer.add_to(route_map)
folium.LayerControl(collapsed=False).add_to(route_map)
route_map

## 4. How often do the turn restrictions change a route?

The cell draws 300 pairs of nodes at random, and computes the route between them on the two networks. It counts the routes that are longer with the turn restrictions. A fixed seed gives the same pairs at each run.

In [7]:
node_points = [{"latitude": latitude, "longitude": longitude} for latitude, longitude in streets.nodes]
generator = random.Random(1)
pairs = 300
longer = 0
for _ in range(pairs):
    origin, destination = generator.sample(node_points, 2)
    km = streets.get_shortest_path(origin_node=origin, destination_node=destination)["length"]
    km_any_turn = streets_any_turn.get_shortest_path(origin_node=origin, destination_node=destination)["length"]
    longer += km > km_any_turn
print(f"{longer} of {pairs} routes are longer with the turn restrictions")

73 of 300 routes are longer with the turn restrictions


## 5. The fastest route

A **metric** gives each edge a weight, under a name. `create_metric` computes the weights from a weight per km: the weight of 1 km. The metric below counts minutes at 30 km/h on every street. `off_graph_weight_per_km` gives the minutes of 1 km on foot, at 5 km/h, on the straight lines between each place and its closest node.

The streets have no class, because `select_streets` returns a direction alone. So the metric gives one speed to every street. A speed for each kind of street needs a class for each kind, and scgraph has no class for a street below `primary`.

In [8]:
streets.create_metric("minutes", weight_per_km=60 / 30, off_graph_weight_per_km=60 / 5)

fastest = streets.get_shortest_path(
    origin_node=COPLEY_SQUARE, destination_node=LOGAN_AIRPORT, metric="minutes", output_legs=True
)
km_on_foot = sum(leg["length"] for leg in fastest["legs"] if leg["edge_mode"] == "off_graph")
print(f"{fastest['length']:.3f} km in {fastest['metric_totals']['minutes']:.1f} minutes, of which {km_on_foot:.3f} km on foot")

7.914 km in 22.3 minutes, of which 0.643 km on foot


With one speed everywhere, the fastest route is the shortest route. Its minutes also count the km on foot at the two ends, at 5 km/h.

## What to remember

- The Overpass API gives the streets of a city and their turn restrictions in one file.
- `GeoGraph.load_from_osm(file, select=...)` loads the ways that `select` keeps. It needs `pip install scgraph[osm]`.
- A route on the geograph makes no turn that a turn restriction of the file forbids. `load_turn_restrictions=False` allows every turn.
- A junction with a turn restriction has several nodes with the same OpenStreetMap id in `external_node_ids`.

The guide [Networks](https://github.com/connor-makowski/scgraph/blob/main/guides/networks.md), section *Turn Restrictions*, gives the rules of the turn restrictions. The notebook *The motorways of a country* loads one kind of road for a whole country.